In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

import sys
from pathlib import Path

# add repo root so swiss_roll_models can be imported from anywhere
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "swiss_roll_models").exists():
        sys.path.insert(0, str(p))
        break

from swiss_roll_models.environment.dataset import generate_swiss_roll
from swiss_roll_models.environment.hilbert_distance import hilbert_analysis as hda

# positive linear model = softplus(theta) ∈ R^D_{>0}
class PositiveLinear(nn.Module):
    def __init__(self, D):
        super().__init__()
        self.theta = nn.Parameter(0.5*torch.ones(D))  # trainable parameters
    def forward(self, X):
        w = F.relu(self.theta)
        return X @ w

    def positive_params_vector(self):
        # Return the current positive parameter vector for Hilbert distance calculation
        return F.relu(self.theta).detach().clone()

In [2]:
def run_experiment_on_subset(
    X_full, y_full, n,
    num_epochs=500,
    lr=1e-2,
    l2_reg=1e-3,
    device="cuda"
):
    N, D = X_full.shape
    idx = torch.randperm(N, device=device)[:n]
    X = X_full[idx]
    y = y_full[idx]

    dataset = TensorDataset(X, y)
    loader = DataLoader(dataset, batch_size=n, shuffle=False)  # full-batch

    model = PositiveLinear(D).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    param_traj = []
    loss_traj = []

    for epoch in range(num_epochs):
        for batch_X, batch_y in loader:
            optimizer.zero_grad()

            # 1. prediction
            y_pred = model(batch_X)

            # 2. MSE part
            mse = F.mse_loss(y_pred, batch_y)

            # 3. L2 regularization: regularize w = softplus(theta) in the positive cone
            w = F.relu(model.theta)
            l2 = (w ** 2).sum()

            # 4. Total loss = mse + λ ||w||^2
            loss = mse + l2_reg * l2

            loss.backward()
            optimizer.step()

        loss_traj.append(loss.item())
        param_traj.append(model.positive_params_vector().cpu())

    w_star = param_traj[-1]

    return {
        "n": n,
        "loss_traj": loss_traj,
        "param_traj": param_traj,
        "w_star": w_star,
    }


In [3]:
def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Using device:", device)

    # ----- Generate Swiss roll -----
    N = 10_000
    D = 32
    X_full, y_full, u, v = generate_swiss_roll(
        n_samples=N,
        D=D,
        noise=0.0,
        device=device,
    )
    print(f"Full dataset: X={X_full.shape}, y={y_full.shape}")

    # ----- Different small sample sizes -----
    n_list = [50, 100, 200, 500]
    num_epochs = 500
    lr = 1e-2
    l2_list = [0.0, 1e-4, 1e-3, 1e-2]

    # all_results[n][l2_reg] = corresponding experiment results
    all_results = {}

    for n in n_list:
        all_results[n] = {}
        print(f"\n================ n = {n} ================")

        for l2_reg in l2_list:
            print(f"\n--- Running experiment (n={n}, l2_reg={l2_reg}) ---")
            res = run_experiment_on_subset(
                X_full,
                y_full,
                n=n,
                num_epochs=num_epochs,
                lr=lr,
                l2_reg=l2_reg,
                device=device,
            )
            all_results[n][l2_reg] = res
            analysis= hda.analysis_distance_on_cone(res["param_traj"], res["w_star"],threshold=1e-6,ifmask=True)
            hilbert_to_final=analysis["hilbert_to_final"]
            hilbert_to_init=analysis["hilbert_to_init"]
            hilbert_between=analysis["hilbert_between"]
            print(f"Final loss: {res['loss_traj'][-1]:.6f}")
            print(f"Initial d_H(w_t, w*): {hilbert_to_final[0]:.6f}")
            print(f"Final   d_H(w_t, w*): {hilbert_to_final[-1]:.6f}")

            hilbert = hilbert_to_final
            ratio_to_prev = [hilbert[i] / hilbert[i-1] if hilbert[i-1] != 0 else float("inf") for i in range(1, len(hilbert))]
            init_dist = hilbert[0]
            ratio_to_init = [d / init_dist if init_dist != 0 else float("inf") for d in hilbert]

            print("First 15 ratio d_H(w_t, w*)/d_H(w_{t-1}, w*):", ratio_to_prev[:15])
            print("First 15 ratio d_H(w_t, w*)/d_H(w_0, w*):", ratio_to_init[:15])
            hil= hilbert_to_final[:15]
            print("First 15 d_H(w_t, w*):", hil)
            print("First 15 d_H(w_t, w_0):", hilbert_to_init[:15])
            print("First 15 d_H(w_{t+1}, w_t):", hilbert_between[:15])
    torch.save(all_results, "swiss_roll_cone_l2_experiments.pt")

    
if __name__ == "__main__":
    main()


Using device: cuda
Full dataset: X=torch.Size([10000, 32]), y=torch.Size([10000])

================ n = 50 ================

--- Running experiment (n=50, l2_reg=0.0) ---
Final loss: 0.000404
Initial d_H(w_t, w*): 5.238538
Final   d_H(w_t, w*): 0.000000
First 15 ratio d_H(w_t, w*)/d_H(w_{t-1}, w*): [0.29128058855939615, 3.4980018432744817, 0.20202102138829284, 2.667407152889068, 0.5302225409326824, 0.9392318325446788, 0.7098521157723618, 0.9392549239931071, 0.7648989235938025, 0.9442156144384843, 0.7995077735654702, 0.9501126911945655, 0.8255817247294541, 0.9562348415626799, 0.8467034100061258]
First 15 ratio d_H(w_t, w*)/d_H(w_0, w*): [1.0, 0.29128058855939615, 1.0189000356908435, 0.20583922590283224, 0.5490570235183635, 0.2911224101268422, 0.2734314347582576, 0.19409588248182166, 0.18230551334783845, 0.13944529092497723, 0.13166642105128057, 0.10526832714804309, 0.10001677360417717, 0.08257202045401192, 0.07895824289635246]
First 15 d_H(w_t, w*): [5.2385382652282715, 1.52588450908660